# Rotten Fruit Detection Dataset Prep - SageMaker Studio

Use this notebook to validate the Roboflow object-detection dataset and generate model-agnostic artifacts. For SSD EfficientDet D5 with the TensorFlow Object Detection API, use the generated `*.record` files and `label_map.pbtxt`.

In [ ]:
import sys
from pathlib import Path

import boto3
import sagemaker
import tensorflow as tf

session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()
bucket = session.default_bucket()

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("Region:", region)
print("Execution role:", role)
print("Default bucket:", bucket)

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_DIR / "src"

S3_DATA_ZIP_URI = None
LOCAL_DATA_ZIP_PATH = PROJECT_DIR / "dataset.zip"
EXTRACT_DIR = PROJECT_DIR / "data"
LOCAL_DATA_DIR = EXTRACT_DIR / "dataset"
OUTPUT_DIR = PROJECT_DIR / "artifacts"
S3_OUTPUT_PREFIX = "rotten-fruit-detection/artifacts"

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Dataset zip:", LOCAL_DATA_ZIP_PATH)
print("Artifacts output:", OUTPUT_DIR)

Expected zip contents:

```text
dataset/
  train/
    image1.jpg
    image2.jpg
    _annotations.csv
  valid/
    image3.jpg
    _annotations.csv
  test/
    image4.jpg
    _annotations.csv
```

In [ ]:
import zipfile


def split_s3_uri(s3_uri):
    if not s3_uri.startswith("s3://"):
        raise ValueError("S3 URI must start with s3://")
    bucket_name, _, key = s3_uri[5:].partition("/")
    return bucket_name, key


def download_s3_file(s3_uri, local_path):
    bucket_name, key = split_s3_uri(s3_uri)
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    boto3.client("s3").download_file(bucket_name, key, str(local_path))


def has_train_test(path):
    child_names = {child.name.lower() for child in Path(path).iterdir() if child.is_dir()}
    return "train" in child_names and "test" in child_names


def find_dataset_root(search_dir):
    search_dir = Path(search_dir)
    if has_train_test(search_dir):
        return search_dir

    for path in search_dir.rglob("*"):
        if path.is_dir() and has_train_test(path):
            return path

    raise ValueError(f"Could not find a folder with train and test inside {search_dir}.")


if S3_DATA_ZIP_URI:
    download_s3_file(S3_DATA_ZIP_URI, LOCAL_DATA_ZIP_PATH)

if LOCAL_DATA_ZIP_PATH.exists():
    with zipfile.ZipFile(LOCAL_DATA_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    LOCAL_DATA_DIR = find_dataset_root(EXTRACT_DIR)
else:
    LOCAL_DATA_DIR = find_dataset_root(LOCAL_DATA_DIR)

print("Dataset root:", LOCAL_DATA_DIR)

In [ ]:
possible_src_dirs = [SRC_DIR, PROJECT_DIR / "src", Path.cwd() / "src", Path.cwd().parent / "src"]
SRC_DIR = next((path for path in possible_src_dirs if path.exists()), None)
if SRC_DIR is None:
    raise FileNotFoundError("Could not find src folder. Check PROJECT_DIR.")

sys.path.insert(0, str(SRC_DIR))
print("Using src folder:", SRC_DIR)

In [ ]:
from train import prepare_training_data

dataset, summary = prepare_training_data(
    data_dir=LOCAL_DATA_DIR,
    output_dir=OUTPUT_DIR,
    export_tensorflow_records=True,
)

summary

In [ ]:
artifacts_s3_uri = session.upload_data(
    path=str(OUTPUT_DIR),
    bucket=bucket,
    key_prefix=S3_OUTPUT_PREFIX,
)

print("Artifacts S3 URI:", artifacts_s3_uri)

Generated artifacts include `class_names.txt`, `dataset_summary.json`, JSON annotations, `label_map.pbtxt`, and `*.record` TFRecord files. For SSD EfficientDet D5, plug the `.record` paths and `label_map.pbtxt` into the TensorFlow Object Detection API pipeline config.